<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/14%E1%84%8C%E1%85%AE%E1%84%8E%E1%85%A1_LangGraph_%E1%84%80%E1%85%B5%E1%84%87%E1%85%A9%E1%86%AB_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧱 14주차 · LangGraph 기본 연습

> LangGraph를 **손에 익히는** 시간입니다. 가장 단순한 그래프부터 직접 만들며,
> State·Node·Edge·분기·반복을 하나씩 연습합니다.

### ✨ 이번 주의 특징
- **3~8번 예제는 LLM 없이 순수 파이썬으로 돌아갑니다 → API 키 없이 바로 실행 가능!**
- 마지막(Part 7)에서만 LLM 노드를 붙여 15주차(에이전트)와 다리를 놓습니다.

### 🎯 학습 흐름
| 파트 | 내용 | API 키 |
| --- | --- | --- |
| 1 | 첫 그래프 (노드 1개) | ❌ 불필요 |
| 2 | 노드 여러 개 (순서대로) | ❌ |
| 3 | State — 덮어쓰기 vs 이어붙이기 (Reducer) | ❌ |
| 4 | 갈림길 (조건 분기) | ❌ |
| 5 | 반복 (루프) | ❌ |
| 6 | 그래프 시각화 | ❌ |
| 7 | LLM 노드 붙이기 (15주차 예고) | ✅ 필요 |

> 💡 위에서부터 순서대로 실행하세요. (Google Colab 기준)

---
## 0. 준비 — 설치

LangGraph만 설치하면 3~6번 예제를 바로 돌릴 수 있습니다.
(langchain-google-genai는 7번 LLM 예제에서만 씁니다.)

In [ ]:
!pip install -q langgraph langchain-google-genai langchain-core

print("✅ 설치 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 13.9 MB/s eta 0:00:00
✅ 설치 완료!


---
## 🌱 Part 1. 첫 그래프 — 노드 1개짜리

글자를 대문자로 바꿔 외치는 노드 하나짜리 그래프입니다.

```
START → [shout] → END
"hello"   대문자+!   "HELLO!"
```

**개념 정리**
- **State** = 그래프가 들고 다니는 공유 메모장 (여기선 `text` 하나)
- **Node** = 상태를 받아 바꿀 부분만 돌려주는 함수
- **Edge** = 노드를 잇는 화살표

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

# ① State 정의 — 공유 메모장의 모양
class State(TypedDict):
    text: str

# ② Node 정의 — text를 대문자로 바꿔 돌려줌
def shout(state: State):
    return {"text": state["text"].upper() + "!"}

# ③ 그래프 조립
builder = StateGraph(State)
builder.add_node("shout", shout)
builder.add_edge(START, "shout")
builder.add_edge("shout", END)
graph = builder.compile()

# ④ 실행
result = graph.invoke({"text": "hello"})
print(result)          # {'text': 'HELLO!'}

{'text': 'HELLO!'}


> 🔧 **연습**: `shout` 노드를 바꿔서 text 뒤에 `" 🎉"`를 붙여 보세요.
> 그리고 `graph.invoke({"text": "langgraph"})`로 실행해 보세요.

---
## 🔗 Part 2. 노드 여러 개 — 순서대로 흐르기

노드를 이으면 상태가 노드를 지나며 **조금씩 바뀝니다.**

```
START → [add_one] → [double] → END
  3         4          8
```

In [ ]:
class State(TypedDict):
    number: int

def add_one(state: State):
    return {"number": state["number"] + 1}

def double(state: State):
    return {"number": state["number"] * 2}

builder = StateGraph(State)
builder.add_node("add_one", add_one)
builder.add_node("double", double)
builder.add_edge(START, "add_one")
builder.add_edge("add_one", "double")   # add_one → double 순서
builder.add_edge("double", END)
graph = builder.compile()

print(graph.invoke({"number": 3}))       # 3 → 4 → 8 → {'number': 8}

{'number': 8}


> 🔧 **연습**: `double` 뒤에 10을 빼는 노드 `minus_ten`을 추가해
> `add_one → double → minus_ten` 순서로 이어 보세요. (3 → 4 → 8 → -2)

---
## 📚 Part 3. State — 덮어쓰기 vs 이어붙이기 (Reducer) ⭐

초보자가 가장 헷갈리는 부분입니다. 천천히 봅시다.

- **기본은 덮어쓰기**: 여러 노드가 같은 필드를 건드리면 **마지막 값만** 남습니다.
- **이어붙이려면 Reducer**: 필드에 `operator.add`를 붙이면 리스트가 **쌓입니다.**

먼저 **덮어쓰기(기본)** 부터 봅시다.

In [ ]:
# ── 덮어쓰기 (Reducer 없음) ──
class State(TypedDict):
    steps: list          # reducer 없음 → 덮어씀

def node_a(state): return {"steps": ["A 실행"]}
def node_b(state): return {"steps": ["B 실행"]}

builder = StateGraph(State)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)
graph = builder.compile()

print(graph.invoke({"steps": []}))    # {'steps': ['B 실행']}  ← A가 사라짐!

{'steps': ['B 실행']}


이번엔 `operator.add`를 붙여 **이어붙이기**로 바꿔봅시다. (필드 정의 한 줄만 다름!)

In [ ]:
import operator
from typing import Annotated

# ── 이어붙이기 (Reducer = operator.add) ──
class State(TypedDict):
    steps: Annotated[list, operator.add]   # ← "이어붙여라"

def node_a(state): return {"steps": ["A 실행"]}
def node_b(state): return {"steps": ["B 실행"]}

builder = StateGraph(State)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)
graph = builder.compile()

print(graph.invoke({"steps": []}))    # {'steps': ['A 실행', 'B 실행']}  ← 쌓임!

{'steps': ['A 실행', 'B 실행']}


> 💡 **다음 주 예고**: 15주차에서 대화 기록을 담을 때 쓰는 `add_messages`도
> 바로 이런 **Reducer의 하나**입니다("메시지를 이어붙여라"). 지금 원리를 잡으면 다음 주가 쉬워집니다.

---
## 🔀 Part 4. 갈림길 — 조건 분기

상황에 따라 **다른 길로 가는** 그래프입니다. 숫자가 짝수면 A, 홀수면 B로 갑니다.

```
              ┌─(짝수)→ [even_node] → END
START →(판단)─┤
              └─(홀수)→ [odd_node]  → END
```

핵심은 **"어디로 갈지 정해주는 함수(router)"** 입니다.

In [ ]:
class State(TypedDict):
    number: int
    result: str

def even_node(state): return {"result": "짝수입니다 ✅"}
def odd_node(state):  return {"result": "홀수입니다 🔷"}

# 라우터: 상태를 보고 "어느 노드로 갈지" 이름표를 반환
def route(state):
    return "even" if state["number"] % 2 == 0 else "odd"

builder = StateGraph(State)
builder.add_node("even_node", even_node)
builder.add_node("odd_node", odd_node)
# START에서 route 결과에 따라 갈림
builder.add_conditional_edges(START, route, {"even": "even_node", "odd": "odd_node"})
builder.add_edge("even_node", END)
builder.add_edge("odd_node", END)
graph = builder.compile()

print(graph.invoke({"number": 4, "result": ""}))   # 짝수입니다
print(graph.invoke({"number": 7, "result": ""}))   # 홀수입니다

{'number': 4, 'result': '짝수입니다 ✅'}
{'number': 7, 'result': '홀수입니다 🔷'}


> 🔧 **연습**: `route`를 바꿔서 "3의 배수면 fizz_node, 아니면 normal_node"로 가게 만들어 보세요.

---
## 🔁 Part 5. 반복 — 루프

조건 분기를 이용하면 **되돌아가는 화살표(반복)** 도 만들 수 있습니다.
0부터 5가 될 때까지 1씩 더합니다.

```
        ┌──────(count < 5)──────┐
        ▼                       │
START → [increment] ──(판단)────┘
             │ (count == 5)
             ▼
            END
```

In [ ]:
class State(TypedDict):
    count: int

def increment(state: State):
    new = state["count"] + 1
    print(f"count: {new}")
    return {"count": new}

# 라우터: 5 미만이면 계속(자기 자신으로), 아니면 끝
def should_continue(state):
    return "continue" if state["count"] < 5 else "end"

builder = StateGraph(State)
builder.add_node("increment", increment)
builder.add_edge(START, "increment")
builder.add_conditional_edges(
    "increment", should_continue,
    {"continue": "increment", "end": END}   # "continue" → 자기 자신으로(루프)
)
graph = builder.compile()

final = graph.invoke({"count": 0})   # count: 1,2,3,4,5 출력
print("최종:", final)

count: 1
count: 2
count: 3
count: 4
count: 5
최종: {'count': 5}


> ⚠️ **무한 루프 주의**: 반복은 반드시 **멈추는 조건**이 필요합니다.
> LangGraph는 안전을 위해 기본 25번까지만 돌고 멈춥니다(재귀 한도).
>
> 🔧 **연습**: 5 대신 10이 될 때까지 반복하도록 바꿔 보세요.

---
## 🖼️ Part 6. 그래프 그림으로 보기

내가 만든 그래프가 어떻게 생겼는지 텍스트(mermaid)로 확인할 수 있습니다.
출력된 내용을 [mermaid.live](https://mermaid.live)에 붙이면 그림으로도 볼 수 있어요.

In [ ]:
# 바로 위(Part 5)에서 만든 루프 그래프의 구조 출력
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	increment(increment)
	__end__([<p>__end__</p>]):::last
	__start__ --> increment;
	increment -. &nbsp;end&nbsp; .-> __end__;
	increment -. &nbsp;continue&nbsp; .-> increment;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



---
## 🤖 Part 7. LLM 노드 붙이기 (15주차로 가는 다리)

노드는 그냥 함수이니, 그 **안에서 LLM을 호출**할 수도 있습니다.
여기서부터는 **API 키가 필요**합니다.

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Google API 키를 입력하세요: ")
print("✅ API 키 설정 완료!")

Google API 키를 입력하세요: ··········
✅ API 키 설정 완료!


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

class QAState(TypedDict):
    question: str
    answer: str

# 노드 안에서 LLM 호출
def ask_llm(state: QAState):
    response = llm.invoke(state["question"])
    return {"answer": response.content}

builder = StateGraph(QAState)
builder.add_node("ask_llm", ask_llm)
builder.add_edge(START, "ask_llm")
builder.add_edge("ask_llm", END)
qa_graph = builder.compile()

result = qa_graph.invoke({"question": "LangGraph를 한 문장으로 설명해줘", "answer": ""})
print("🤖", result["answer"])

🤖 [{'type': 'text', 'text': 'LangGraph는 LLM을 활용한 복잡하고 상태 유지(stateful)가 가능한 다중 에이전트 시스템을 순환형 그래프 구조로 설계하고 구현할 수 있게 해주는 프레임워크입니다.', 'extras': {'signature': 'EjQKMgERTTIP7eijOE7XjOyEBGPaNCVrL/3mQ5CYXwffHAXhsm81RXugSvENRHDWx48MhqhH'}}]


이 "LLM을 부르는 노드"에 **갈림길·반복·도구**를 더하면 → 다음 주(15주차)의 **에이전트**가 됩니다.
오늘 배운 State·Node·Edge·분기·반복이 그대로 재료가 됩니다.

---
## 🎉 정리

| 배운 것 | 핵심 |
| --- | --- |
| **State** | 노드들이 함께 보는 공유 메모장 (`TypedDict`) |
| **Node** | 상태를 받아 바뀔 부분만 돌려주는 함수 |
| **Edge** | 노드를 잇는 화살표 (`add_edge`) |
| **Reducer** | 덮어쓰기(기본) vs 이어붙이기(`operator.add`) |
| **조건 분기** | 라우터 함수 + `add_conditional_edges` |
| **반복** | 자기 자신으로 되돌아오는 조건 분기 (멈춤 조건 필수) |

이 기본기가 **15주차 에이전트**의 재료가 됩니다.

## ✏️ 종합 과제

배운 걸 합쳐, **숫자 맞히기 판정 그래프**를 만들어 보세요.

- State: `{"number": int, "log": Annotated[list, operator.add]}`
- 노드 `judge`: number가 100보다 크면 log에 "너무 큼", 작으면 "너무 작음", 같으면 "정답!" 추가
- (심화) 조건 분기로 "정답이면 END, 아니면 다시 판정"하는 루프 구조로 확장

> 💡 막히면 Part 3(Reducer) + Part 4(분기) + Part 5(루프)를 조합하세요.

In [ ]:
# ✏️ 여기서 자유롭게 실험!
# 예) Part 2의 숫자 파이프라인에 노드를 더 추가해 보세요.
